# Segger I/O Workflows

This notebook collects Segger input/output scenarios and SpatialData I/O examples.

## Requirements

- Segger is GPU-only and requires RAPIDS (CuPy/cuDF/cuML/cuGraph/cuSpatial).
- Optional extras:
  - `segger[spatialdata-io]` for platform-specific SpatialData readers.
  - `segger[spatialdata]` for full SpatialData I/O.
  - `segger[spatialdata-all]` for SpatialData + SOPA.

## Scenario 1: Raw platform input (auto-detect)

Segger auto-detects Xenium, CosMx, and MERSCOPE based on folder contents.

```bash
# Xenium (10x)
segger segment -i /path/to/xenium_sample -o /path/to/out

# CosMx (NanoString)
segger segment -i /path/to/cosmx_sample -o /path/to/out

# MERSCOPE (Vizgen)
segger segment -i /path/to/merscope_sample -o /path/to/out
```

Notes:
- MERSCOPE raw boundary loading is not implemented yet. Provide boundaries via SpatialData or downstream steps.
- Use `--input-format raw` to force raw platform parsing.

## Scenario 2: SpatialData Zarr input

If you already have a SpatialData `.zarr` store, Segger can read it directly.

```bash
segger segment -i /path/to/experiment.zarr -o /path/to/out
# or explicitly
segger segment -i /path/to/experiment.zarr -o /path/to/out --input-format spatialdata
```

Optional keys:
- `--spatialdata-points-key` for transcripts
- `--spatialdata-shapes-key` for boundaries

## Scenario 3: Output formats (segment)

`--output-format` supports: `segger_raw`, `merged`, `spatialdata`, `anndata`, `all`.

```bash
# Default (predictions parquet)
segger segment -i /path/to/data -o /path/to/out

# Merged transcripts
segger segment -i /path/to/data -o /path/to/out --output-format merged

# SpatialData Zarr
segger segment -i /path/to/data -o /path/to/out --output-format spatialdata

# AnnData
segger segment -i /path/to/data -o /path/to/out --output-format anndata

# All formats
segger segment -i /path/to/data -o /path/to/out --output-format all
```

## Scenario 4: SpatialData boundaries

Boundary methods: `input`, `convex_hull`, `delaunay`, `voxel`, `skip`.

Note: `delaunay` can be slow on large datasets; consider `--boundary-n-jobs` (or higher `--num-workers`) to parallelize.
SpatialData output is SOPA-compatible by default (points include a `cell_id` alias).

```bash
# Use input boundaries if available
segger segment -i /path/to/data -o /path/to/out \
  --output-format spatialdata --boundary-method input

# Delaunay boundaries with parallel workers
segger segment -i /path/to/data -o /path/to/out \
  --output-format spatialdata --boundary-method delaunay --boundary-n-jobs 8

# Voxel boundaries (fast, axis-aligned)
segger segment -i /path/to/data -o /path/to/out \
  --output-format spatialdata --boundary-method voxel --boundary-voxel-size 2.0
```


## Scenario 5: AnnData output

```bash
segger segment -i /path/to/data -o /path/to/out --output-format anndata
```

Output: `segger_segmentation.h5ad` with a cell x gene matrix.
When writing SpatialData, the AnnData table can be embedded as a table.

## Scenario 6: Export to Xenium Explorer

```bash
# If your segmentation parquet already includes x/y/compartment columns
segger export -s /path/to/out/segger_segmentation.parquet \
  -i /path/to/xenium_experiment -o /path/to/export --format xenium_explorer

# If it only has assignments, Segger will merge with transcripts
segger export -s /path/to/out/segger_segmentation.parquet \
  -i /path/to/xenium_experiment -o /path/to/export --format xenium_explorer

# Use convex hull boundaries instead of Delaunay
segger export -s /path/to/out/segger_segmentation.parquet \
  -i /path/to/xenium_experiment -o /path/to/export --format xenium_explorer \
  --boundary-method convex_hull
```

Notes:
- `--boundary-method` applies to Xenium Explorer export.
- `--boundary-method skip` is not supported for Xenium Explorer.
- `--format xenium` is deprecated; use `xenium_explorer`.
- `--boundary-method delaunay` can be accelerated with `--boundary-voxel-size` (voxel downsampling), which often improves speed and stability.
- If Xenium Explorer shows missing cells or a "could not be opened" warning on first load, try closing other Explorer windows / freeing RAM and reopen.
- Xenium export accepts either **raw Segger output** (row_index + segger_cell_id) or **merged parquet**; if x/y are missing it will auto-merge with transcripts from `-i`.
- SpatialData input (`--input-format spatialdata`) is also supported for Xenium export; transcripts/boundaries are loaded from the SpatialData object. If you pass a SpatialData `.zarr` as `-s`, shapes are required (export will error if missing).


## Scenario 7: Export merged / SpatialData / AnnData from segmentation parquet

```bash
# Merged transcripts
segger export -s /path/to/out/segger_segmentation.parquet \
  -i /path/to/raw_or_zarr -o /path/to/export --format merged

# SpatialData Zarr (with optional boundary generation)
segger export -s /path/to/out/segger_segmentation.parquet \
  -i /path/to/raw_or_zarr -o /path/to/export --format spatialdata \
  --boundary-method delaunay --boundary-n-jobs 8

# AnnData
segger export -s /path/to/out/segger_segmentation.parquet \
  -i /path/to/raw_or_zarr -o /path/to/export --format anndata
```

Notes:
- `-i` can be a raw platform folder or a SpatialData `.zarr` store.
- Use `--input-format spatialdata` to force SpatialData parsing.

## Scenario 8: SpatialData-only readers/writers

If you only need platform-specific readers without full SpatialData, install:

```bash
pip install segger[spatialdata-io]
```

For full SpatialData workflows, install:

```bash
pip install segger[spatialdata]
```

For SpatialData + SOPA, install:

```bash
pip install segger[spatialdata-all]
```


# SpatialData I/O with Segger

This notebook demonstrates how to:
1. Read spatial transcriptomics data from SpatialData Zarr stores
2. Run Segger segmentation (simulated)
3. Export results to SpatialData-compatible Zarr format
4. Validate compatibility with SOPA workflows

## Setup

## Input/Output Formats (CLI + API)

Segger accepts input in raw platform formats or SpatialData Zarr, and can write multiple output formats:

- **input_format**: `auto` (default), `raw`, `spatialdata`
- **output_format**: `segger_raw`, `merged`, `spatialdata`, `anndata`, `all`

CLI examples:
```bash
segger segment -i /path/to/data -o /path/to/out --input-format raw --output-format segger_raw
segger segment -i /path/to/experiment.zarr -o /path/to/out --input-format spatialdata --output-format spatialdata
segger segment -i /path/to/data -o /path/to/out --output-format all
```

Notes:
- SpatialData input/output requires `segger[spatialdata]` (or `segger[spatialdata-all]`)
- SpatialData output includes an AnnData table in `sdata.tables["cell_table"]`


In [ ]:
import sys
sys.path.insert(0, '../src')

import polars as pl
import geopandas as gpd
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path

## 1. Create Sample Data

First, let's create some synthetic Xenium-like data to work with.

In [ ]:
from segger.datasets import create_synthetic_xenium

# Create synthetic Xenium data
transcripts, cells, boundaries = create_synthetic_xenium(
    n_cells=100,
    transcripts_per_cell=30,
    seed=42,
)

print(f"Generated {len(transcripts):,} transcripts")
print(f"Generated {len(cells):,} cells")
print(f"Generated {len(boundaries):,} cell boundaries")
print(f"\nTranscript columns: {transcripts.columns}")

In [ ]:
# Visualize the data
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot transcripts
ax1 = axes[0]
tx_pd = transcripts.to_pandas()
ax1.scatter(tx_pd['x_location'], tx_pd['y_location'], s=1, alpha=0.5)
ax1.set_xlabel('X (microns)')
ax1.set_ylabel('Y (microns)')
ax1.set_title('Transcript Positions')
ax1.set_aspect('equal')

# Plot boundaries
ax2 = axes[1]
boundaries.plot(ax=ax2, facecolor='lightblue', edgecolor='navy', alpha=0.5)
ax2.set_xlabel('X (microns)')
ax2.set_ylabel('Y (microns)')
ax2.set_title('Cell Boundaries')
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

## 2. Write to SpatialData Zarr Format (Lightweight)

Segger includes a lightweight SpatialData writer (`segger.io.spatialdata_zarr`) that creates Zarr stores compatible with the scverse ecosystem and SOPA, without requiring the full `spatialdata` package.


In [ ]:
from segger.io.spatialdata_zarr import (
    SpatialDataZarrWriter,
    SpatialDataZarrReader,
    write_spatialdata_zarr,
    read_spatialdata_zarr,
    get_spatialdata_info,
)

# Create a temporary directory for output
output_dir = Path(tempfile.mkdtemp())
zarr_path = output_dir / "experiment.zarr"

# Standardize column names
tx_standard = transcripts.rename({
    'x_location': 'x',
    'y_location': 'y',
    'z_location': 'z',
})

# Write to SpatialData format
write_spatialdata_zarr(
    tx_standard,
    zarr_path,
    shapes=boundaries,
    points_key="transcripts",
    shapes_key="cells",
)

print(f"Wrote SpatialData to: {zarr_path}")
print(f"\nStore info: {get_spatialdata_info(zarr_path)}")

## 3. Read from SpatialData Zarr Format

The reader can load data from any SpatialData-compatible Zarr store.

In [ ]:
# Using the class-based reader
reader = SpatialDataZarrReader(zarr_path)

print("Available elements:")
print(f"  Points: {reader.points_keys}")
print(f"  Shapes: {reader.shapes_keys}")

# Read data
tx_loaded = reader.read_points("transcripts")
shapes_loaded = reader.read_shapes("cells")

print(f"\nLoaded {len(tx_loaded):,} transcripts")
print(f"Loaded {len(shapes_loaded):,} cell boundaries")

In [ ]:
# Or use the convenience function
tx_loaded, shapes_loaded = read_spatialdata_zarr(zarr_path)

print(f"Loaded transcripts shape: {tx_loaded.shape}")
print(f"Loaded shapes shape: {shapes_loaded.shape}")

## 4. Simulate Segger Segmentation

Now let's simulate a Segger segmentation run. In practice, you would run:
```bash
segger segment -i data/ -o output/
```

Here we'll use the sample output generator.

In [ ]:
from segger.datasets import create_sample_segger_output, create_merged_output

# Generate sample Segger outputs
tx_data, predictions, cell_boundaries = create_sample_segger_output(
    n_cells=100,
    transcripts_per_cell=30,
    unassigned_rate=0.1,  # 10% unassigned
    seed=42,
)

print("Segger Predictions Format:")
print(predictions.head())

# Statistics
n_assigned = (predictions['segger_cell_id'] >= 0).sum()
n_unassigned = (predictions['segger_cell_id'] < 0).sum()
print(f"\nAssigned: {n_assigned:,} ({100*n_assigned/len(predictions):.1f}%)")
print(f"Unassigned: {n_unassigned:,} ({100*n_unassigned/len(predictions):.1f}%)")

In [ ]:
# Create merged output (transcripts + predictions)
merged = create_merged_output(tx_data, predictions)

print("Merged Output Format:")
print(merged.head())
print(f"\nColumns: {merged.columns}")

## 5. Export Segger Results to All Formats

Segger can write multiple outputs from the same predictions:
- `segger_raw`: predictions parquet
- `merged`: transcripts + assignments
- `spatialdata`: SpatialData Zarr (with optional tables)
- `anndata`: `.h5ad` cell x gene matrix

### 5a. Full export writers (SpatialData + AnnData)

This uses `segger.export` writers. SpatialData output requires `segger[spatialdata]` (or `segger[spatialdata-all]`).


In [ ]:
from segger.export import SeggerRawWriter, MergedTranscriptsWriter, AnnDataWriter, SpatialDataWriter

pred_path = SeggerRawWriter().write(
    predictions=predictions,
    output_dir=output_dir,
    output_name="predictions.parquet",
)

merged_path = MergedTranscriptsWriter().write(
    predictions=predictions,
    output_dir=output_dir,
    transcripts=tx_data,
    output_name="transcripts_segmented.parquet",
)

ad_path = AnnDataWriter().write(
    predictions=predictions,
    output_dir=output_dir,
    transcripts=tx_data,
    output_name="segger_segmentation.h5ad",
)

sdata_path = SpatialDataWriter().write(
    predictions=predictions,
    output_dir=output_dir,
    transcripts=tx_data,
    boundaries=cell_boundaries,
    output_name="segmentation_full.zarr",
)

print("Wrote:")
print(f"  segger_raw: {pred_path}")
print(f"  merged: {merged_path}")
print(f"  anndata: {ad_path}")
print(f"  spatialdata: {sdata_path}")


In [ ]:
import spatialdata

sdata = spatialdata.read_zarr(sdata_path)
print("SpatialData tables:", list(sdata.tables.keys()))

# AnnData table with cell x gene counts
cell_table = sdata.tables["cell_table"]
cell_table


### 5b. Lightweight SpatialData Zarr writer (parquet only)

If you only need a lightweight Zarr store from parquet data, use `segger.io.spatialdata_zarr`.


In [ ]:
# Export to SpatialData
segmentation_zarr_light = output_dir / "segmentation_light.zarr"

write_spatialdata_zarr(
    merged,
    segmentation_zarr_light,
    shapes=cell_boundaries,
    points_key="transcripts",
    shapes_key="cells",
)

print(f"Exported segmentation to: {segmentation_zarr_light}")
print(f"\nStore info: {get_spatialdata_info(segmentation_zarr_light)}")


In [ ]:
# Verify the export
reader = SpatialDataZarrReader(segmentation_zarr_light)
tx_exported = reader.read_points()

print("Exported transcripts columns:")
print(tx_exported.columns)

# Check segmentation columns are present
assert 'segger_cell_id' in tx_exported.columns
assert 'segger_similarity' in tx_exported.columns
print("\n✓ Segmentation columns present in exported SpatialData")


## 6. Visualize Segmentation Results

In [ ]:
# Visualize assigned vs unassigned transcripts
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

merged_pd = merged.to_pandas()
assigned = merged_pd[merged_pd['segger_cell_id'] >= 0]
unassigned = merged_pd[merged_pd['segger_cell_id'] < 0]

# Left: All transcripts colored by assignment
ax1 = axes[0]
ax1.scatter(assigned['x'], assigned['y'], s=2, c='blue', alpha=0.5, label='Assigned')
ax1.scatter(unassigned['x'], unassigned['y'], s=5, c='red', alpha=0.8, label='Unassigned')
ax1.set_xlabel('X (microns)')
ax1.set_ylabel('Y (microns)')
ax1.set_title('Transcript Assignment Status')
ax1.legend()
ax1.set_aspect('equal')

# Right: Transcripts colored by cell ID
ax2 = axes[1]
scatter = ax2.scatter(
    assigned['x'], assigned['y'], 
    s=2, 
    c=assigned['segger_cell_id'], 
    cmap='tab20',
    alpha=0.7
)
ax2.set_xlabel('X (microns)')
ax2.set_ylabel('Y (microns)')
ax2.set_title('Transcripts by Cell ID')
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize similarity scores
fig, ax = plt.subplots(figsize=(10, 4))

similarities = merged_pd[merged_pd['segger_similarity'] > 0]['segger_similarity']
ax.hist(similarities, bins=50, edgecolor='white', alpha=0.7)
ax.axvline(similarities.median(), color='red', linestyle='--', label=f'Median: {similarities.median():.3f}')
ax.set_xlabel('Similarity Score')
ax.set_ylabel('Count')
ax.set_title('Distribution of Assignment Similarity Scores')
ax.legend()

plt.tight_layout()
plt.show()

## 7. Save All Outputs

Use the `save_sample_outputs` function to generate a complete set of sample files.

In [ ]:
from segger.datasets import save_sample_outputs

sample_output_dir = output_dir / "sample_outputs"
paths = save_sample_outputs(
    sample_output_dir,
    n_cells=50,
    transcripts_per_cell=20,
    include_spatialdata=True,
)

print("Generated sample outputs:")
for name, path in paths.items():
    print(f"  {name}: {path}")

## 8. SOPA Compatibility

The exported SpatialData Zarr stores are SOPA-compatible by default. Key conventions:

- **shapes["cells"]**: Cell polygons with `cell_id` column
- **points["transcripts"]**: Transcripts with `cell_id` assignment column (alias of `segger_cell_id`)
- Coordinate systems use identity transforms

To use with SOPA:
```python
import sopa
import spatialdata

# Load Segger output
sdata = spatialdata.read_zarr("segmentation.zarr")

# Continue with SOPA analysis
sopa.aggregate(sdata, ...)
```


In [ ]:
# Cleanup
import shutil
shutil.rmtree(output_dir)
print("Cleaned up temporary files")

## Summary

This notebook demonstrated:

1. **Creating synthetic data** with `create_synthetic_xenium()`
2. **Writing to SpatialData (lightweight)** with `write_spatialdata_zarr()`
3. **Reading from SpatialData** with `read_spatialdata_zarr()` or `SpatialDataZarrReader`
4. **Simulating Segger output** with `create_sample_segger_output()`
5. **Exporting segmentation results** to all formats (`segger_raw`, `merged`, `spatialdata`, `anndata`)
6. **AnnData tables inside SpatialData** via `sdata.tables["cell_table"]`
7. **SOPA compatibility** for downstream analysis

### Key Functions

| Function | Purpose |
|----------|------|
| `write_spatialdata_zarr()` | Lightweight Zarr writer (parquet input) |
| `read_spatialdata_zarr()` | Read transcripts + shapes from Zarr |
| `SpatialDataZarrReader` | Class-based reader with metadata |
| `SpatialDataZarrWriter` | Class-based writer with incremental writes |
| `SeggerRawWriter` | Write raw predictions parquet |
| `MergedTranscriptsWriter` | Write transcripts + assignments |
| `AnnDataWriter` | Write `.h5ad` cell x gene matrix |
| `SpatialDataWriter` | Full SpatialData export (includes tables) |
